<a href="https://colab.research.google.com/github/Doufoungognon/portfolio/blob/main/Transformer_LLM_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Занятие содержит:
1. Материалы кафедры ММП факультета ВМК МГУ (раздел про Named Entity Recognition).

2. [**Курс по LLM от Huggingface**](https://huggingface.co/learn/llm-course/chapter7/2)

## **Большие языковые модели**


**Системный промт (System Prompt)** - cкрытая инструкция, задающая роль, стиль, границы и контекст для *всех ответов модели* (например, "Ты — полезный ассистент, отвечающий строго по документации...").

**Пользовательский запрос (User Prompt)** - непосредственный вопрос или задача от пользователя.

Чтобы контролировать баланс между креативностью и точностью ответов, используются ключевые гиперпараметры:

- **Температура (temperature)** регулирует "уровень неопределенности" модели.

 *Низкая (~0.1)*: Ответы детерминированы, консервативны и максимально соответствуют наиболее вероятным паттернам в данных. Идеально для фактологических справок, анализа.

 *Высокая (~0.8-1.0)*: Ответы становятся более случайными, разнообразными и креативными. Подходит для генерации идей, творческих текстов.

- **Top-p** - порог вероятности, необходимый для включения токена в набор кандидатов для генерации: отбирается минимальный набор (ядро) наиболее вероятных токенов, сумма вероятностей которых больше или равна заданному порогу $p$.

*Низкий top-p (0.5)*: Узкий, предсказуемый выбор слов.

*Высокий top-p (0.9)*: Широкий выбор вариантов, что увеличивает разнообразие.



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM-1.7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

### Пайплайн генерации ответа от большой языковой модели в инструкция-ответ, используя библиотеку Hugging Face Transformers.

In [ ]:
messages = [{"role": "user", "content": "What is the capital of France?"}]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print(input_text)
inputs = tokenizer.encode(input_text, return_tensors="pt")
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
What is the capital of France?<|im_end|>

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris, which is located in the Île-de-France region of the country.<|im_end|>


In [ ]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris, which is located in the Île-de-France region of the country.<|im_end|>


In [ ]:
outputs = model.generate(inputs, max_new_tokens=50, temperature=1., top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris, which is located in the northwestern part of the country.<|im_end|>


Уменьшим температуру модели:

In [ ]:
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.7, top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is the city of Paris.<|im_end|>


### RAG + LLM

**RAG (Retrieval Augmented Generation)** системы объединяют два ключевых компонента **извлечение релевантной информации** из существующих баз данных и **генерацию ответов** с использованием языковых моделей. Иными словами, RAG позволяет отвечать на вопросы, опираясь на тексты из внешней базы знаний, с указанием источников.

Принцип работы RAG-систем обычно включает следующие этапы:

Общая схема работы RAG-систем следующая:

1)
Разбиение корпуса на **чанки**. Весь доступный текстовый массив предварительно разбивается на небольшие фрагменты — так называемые чанки. Чанком можно считать, например, последовательность из $n$ предложений. Существует множество подходов к разбиению текста, и выбор метода зависит от конкретной задачи.

2)
Поиск релевантных фрагментов. Запрос, поступивший от пользователя, преобразуется в векторное представление. Затем, с помощью методов поиска ближайших соседей (например, KNN), из корпуса выбираются $k$ чанков, наиболее соответствующих запросу по смыслу.

**Dense Passage Retrieval (DPR)** — это метод семантического поиска, специально разработанный для интеграции с RAG-системами.

Система `DPR` состоит из двух отдельных энкодеров:

`Question Encoder` — преобразует вопрос пользователя в векторное представление.

`Context Encoder` — преобразует каждый чанк текста (например, абзац из Википедии) в отдельный вектор.

После этого `DPR` находит наиболее релевантные документы с помощью поиска ближайших векторов (обычно через `dot product` или `cosine similarity`).

Популярные реализации DPR:

- `facebook/dpr-question_encoder-single-nq-base`

- `facebook/dpr-ctx_encoder-single-nq-base`

Обе модели доступны в Hugging Face Transformers и легко интегрируются в RAG-системы.

In [ ]:
import numpy as np
import torch

from sklearn.neighbors import NearestNeighbors

from transformers import AutoTokenizer, AutoModel, DPRContextEncoder, pipeline

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# Model for encoding the context
tokenizer = AutoTokenizer.from_pretrained("facebook/dpr-ctx_encoder-multiset-base", device = device)
model = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-multiset-base")

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-ctx_encoder-multiset-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# Model for encoding the question
q_tokenizer = AutoTokenizer.from_pretrained("facebook/dpr-question_encoder-multiset-base", device = device)
q_model = AutoModel.from_pretrained("facebook/dpr-question_encoder-multiset-base")

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-multiset-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Будем использовать *Google Flan* `(например, google/flan-t5-base`) в качестве языковой модели для суммаризации. Flan — это модель для выполнения инструкций (instruction-tuned), а не чат-модель.

In [ ]:
# Model for context question answering
pipe = pipeline("text2text-generation", model="google/flan-t5-base", device = device)

Device set to use cuda


OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 3181 has 14.73 GiB memory in use. Of the allocated memory 14.50 GiB is allocated by PyTorch, and 102.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# векторизация чанков
def vectorize_chunks(chunks, tokenizer, model):
  tokenized = tokenizer(chunks, add_special_tokens=True, return_token_type_ids=False, padding=True)
  input_ids = torch.tensor(tokenized.input_ids)
  attention_mask = torch.tensor(tokenized.attention_mask)
  model = model.to(device)

  last_hidden_states = []
  with torch.no_grad():
    for i in tqdm(range(0, len(input_ids), 100)):
      tmp1 = input_ids[i:i+100].to(device)
      tmp2 = attention_mask[i:i+100].to(device)
      last_hidden_states.append(model(tmp1, attention_mask=tmp2).pooler_output.detach().cpu())

  features = torch.cat(last_hidden_states, dim=0)
  print(f"Done\nShape of features array: {features.shape}")

  return features

In [ ]:
# векторизация запроса
def vectorize_question(question, q_tokenizer, q_model):
  q_tokenized = q_tokenizer(question, add_special_tokens=True, return_token_type_ids=False)
  id = torch.tensor([q_tokenized.input_ids])
  mask = torch.tensor([q_tokenized.attention_mask])
  outp = q_model(id, attention_mask=mask).pooler_output.detach()
  return outp[0]

In [ ]:
def find_nearest_chunks (features, question, n_neighbors=5):
    features.numpy()
    nn_model = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine')
    nn_model.fit(features)
    target_vector = question.reshape(1,-1)
    distances, indices = nn_model.kneighbors(target_vector)
    return indices[0].tolist()

In [ ]:
# получение ответа LLM
def answer(question, chunks, features, pipe, NN, q_tokenizer, q_model):
  question_vector = vectorize_question(question, q_tokenizer, q_model)
  indices = find_nearest_chunks(features, question_vector, n_neighbors=NN)
  relevant = [chunks[i] for i in indices]
  prompt = "Source: " + " ".join(relevant[:5]) + "\n\nUsing the source answer the question: " + question
  answer = pipe(prompt)[0]["generated_text"]
  print(f"Q: {question}")
  print(f"A: {answer}")
  print()
  print("Based on:")
  for line in relevant:
    print(line)

In [ ]:
text = """
At the lake shore there was another rowboat drawn up. The two Indians stood waiting.

Nick and his father got in the stern of the boat and the Indians shoved it off and one of them got in to row. Uncle George sat in the stern of the camp rowboat. The young Indian shoved the camp boat off and got in to row Uncle George.

The two boats started off in the dark. Nick heard the oar-locks of the other boat quite a way ahead of them in the mist. The Indians rowed with quick choppy strokes. Nick lay back with his father's arm around him. It was cold on the water. The Indian who was rowing them was working very hard, but the other boat moved further ahead in the mist all the time.

"Where are we going, Dad?" Nick asked.

"Over to the Indian camp. There is an Indian lady very sick."

"Oh," said Nick.

Across the bay they found the other boat beached. Uncle George was smoking a cigar in the dark. The young Indian pulled the boat way up the beach. Uncle George gave both the Indians cigars.

They walked up from the beach through a meadow that was soaking wet with dew, follow虹ng the young Indian who carried a lantern. Then they went into the woods and followed a trail that led to the logging road that ran back into the hills. It was much lighter on the logging road as the timber was cut away on both sides. The young Indian stopped and blew out his lantern and they all walked on along the road.

They came around a bend and a dog came out barking. Ahead were the lights of the shanties where the Indian bark-peelers lived. More dogs rushed out at them. The two Indians sent them back to the shanties. In the shanty nearest the road there was a light in the window. An old woman stood in the doorway holding a lamp.

Inside on a wooden bunk lay a young Indian woman. She had been trying to have her baby for two days. All the old women in the camp had been helping her. The men had moved off up the road to sit in the dark and smoke out of range of the noise she made. She screamed just as Nick and the two Indians followed his father and Uncle George into the shanty. She lay in the lower bunk, very big under a quilt. Her head was turned to one side. In the upper bunk was her husband. He had cut his foot very badly with an ax three days before. He was smoking a pipe. The room smelled very bad.

Nick's father ordered some water to be put on the stove, and while it was heating he spoke to Nick.

"This lady is going to have a baby, Nick," he said.

"I know," said Nick.

"You don't know," said his father. "Listen to me. What she is going through is called being in labor. The baby wants to be born and she wants it to be born. All her muscles are trying to get the baby born. That is what is happening when she screams."

"I see," Nick said.

Just then the woman cried out.

"Oh, Daddy, can't you give her something to make her stop screaming?" asked Nick.

"No. I haven't any anæsthetic," his father said. "But her screams are not important. I don't hear them because they are not important."

The husband in the upper bunk rolled over against the wall.

The woman in the kitchen motioned to the doctor that the water was hot. Nick's father went into the kitchen and poured about half of the water out of the big kettle into a basin. Into the water left in the kettle he put several things he unwrapped from a handkerchief.

"Those must boil," he said, and began to scrub his hands in the basin of hot water with a cake of soap he had brought from the camp. Nick watched his father's hands scrubbing each other with the soap. While his father washed his hands very carefully and thoroughly, he talked.

"You see, Nick, babies are supposed to be born head first but sometimes they're not. When they're not they make a lot of trouble for every苑ody. Maybe I'll have to operate on this lady. We'll know in a little while."

When he was satisfied with his hands he went in and went to work.

"Pull back that quilt, will you, George?" he said. "I'd rather not touch it."

Later when he started to operate Uncle George and three Indian men held the woman still. She bit Uncle George on the arm and Uncle George said, "Damn squaw bitch!" and the young Indian who had rowed Uncle George over laughed at him. Nick held the basin for his father. It all took a long time.

His father picked the baby up and slapped it to make it breathe and handed it to the old woman.

"See, it's a boy, Nick," he said. "How do you like being an interne?"

Nick said, "All right." He was looking away so as not to see what his father was doing.

"There. That gets it," said his father and put something into the basin.

Nick didn't look at it.

"Now," his father said, "there's some stitches to put in. You can watch this or not, Nick, just as you like. I'm going to sew up the incision I made."

Nick did not watch. His curiosity has been gone for a long time.

His father finished and stood up. Uncle George and the three Indian men stood up. Nick put the basin out in the kitchen.

Uncle George looked at his arm. The young Indian smiled reminiscently.

"I'll put some peroxide on that, George," the doctor said.

He bent over the Indian woman. She was quiet now and her eyes were closed. She looked very pale. She did not know what had become of the baby or anything.

"I'll be back in the morning," the doctor said, standing up. "The nurse should be here from St. Ignace by noon and she'll bring everything we need."

He was feeling exalted and talkative as football players are in the dressing room after a game.

"That's one for the medical journal, George," he said. "Doing a Caesarian with a jack-knife and sewing it up with nine-foot, tapered gut leaders."

Uncle George was standing against the wall, looking at his arm.

"Oh, you're a great man, all right," he said.

"Ought to have a look at the proud father. They're usually the worst sufferers in these little affairs," the doctor said. "I must say he took it all pretty quietly."

He pulled back the blanket from the Indian's head. His hand came away wet. He mounted on the edge of the lower bunk with the lamp in one hand and looked in. The Indian lay with his face toward the wall. His throat had been cut from ear to ear. The blood had flowed down into a pool where his body sagged the bunk. His head rested on his left arm. The open razor lay, edge up, in the blankets.

"Take Nick out of the shanty, George," the doctor said.

There was no need of that. Nick, standing in the door of the kitchen, had a good view of the upper bunk when his father, the lamp in one hand, tipped the Indian's head back.

It was just beginning to be daylight when they walked along the logging road back toward the lake.

"I'm terribly sorry I brought you along, Nickie," said his father, all his post-operative exhilaration gone. "It was an awful mess to put you through."

"Do ladies always have such a hard time hav虹ng babies?" Nick asked.

"No, that was very, very exceptional."

"Why did he kill himself, Daddy?"

"I don't know, Nick. He couldn't stand things, I guess."

"Do many men kill themselves, Daddy?"

"Not very many, Nick."

"Do many women?"

"Hardly ever."

"Don't they ever?"

"Oh, yes. They do sometimes."

"Daddy?"

"Yes."

"Where did Uncle George go?"

"He'll turn up all right."

"Is dying hard, Daddy?"

"No, I think it's pretty easy, Nick. It all depends."

They were seated in the boat, Nick in the stern, his father rowing. The sun was coming up over the hills. A bass jumped, making a circle in the water. Nick trailed his hand in the water. It felt warm in the sharp chill of the morning.

In the early morning on the lake sitting in the stern of the boat with his father rowing, he felt quite sure that he would never die.
"""

In [ ]:
chunks = [line for line in text.split('\n')]
not_empty_chunks = chunks[1::2]

In [ ]:
vectorized_chunks = vectorize_chunks(not_empty_chunks, tokenizer, model)

OutOfMemoryError: CUDA out of memory. Tried to allocate 90.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 3181 has 14.73 GiB memory in use. Of the allocated memory 14.50 GiB is allocated by PyTorch, and 102.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 3 наиболее близких чанка
NN = 3

In [ ]:
question = "How many indians stood waiting?"

answer(question, not_empty_chunks, vectorized_chunks, pipe, NN, q_tokenizer, q_model)

In [ ]:
question = "On what young Indian woman laid?"

answer(question, not_empty_chunks, vectorized_chunks, pipe, NN, q_tokenizer, q_model)

## **Выделение сущностей**

**Named Entity Recognition (NER)** — Распознавание Именованных Сущностей.

*Цель задачи*:

Для каждого токена (слова, подслова или символа) необходимо определить, относится ли он к сущности и к какой именно категории.

Сущностями могут быть, например:

- PER (Person) — имена людей: Elon Musk, Александр Пушкин

- LOC (Location) — географические объекты: Paris, Москва

- ORG (Organization) — компании, учреждения: Google, ООН

- MISC (Miscellaneous) — прочие сущности: названия событий, произведений и т.д.

In [ ]:

! pip install transformers[torch] datasets autoawq evaluate seqeval optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 13.6 MB/s eta 0:00:00


In [ ]:
!pip freeze > requirements.txt
!cat requirements.txt | grep torch

torch==2.9.0+cu126
torchao==0.10.0
torchaudio==2.9.0+cu126
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.24.0+cu126


In [ ]:
!cat requirements.txt | grep transformers
!cat requirements.txt | grep datasets
!cat requirements.txt | grep autoawq
!cat requirements.txt | grep evaluate
!cat requirements.txt | grep seqeval
!cat requirements.txt | grep optuna

sentence-transformers==5.1.2
transformers==4.57.3
datasets==4.0.0
tensorflow-datasets==4.9.9
vega-datasets==0.9.0
autoawq==0.2.9
evaluate==0.4.6
seqeval==1.2.2
optuna==4.6.0


In [ ]:
import gc
import json
import random
from typing import Any, Union
from pathlib import Path

import numpy as np
import numpy.typing as npt

import torch
import datasets
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TextGenerationPipeline,
    pipeline,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)
from transformers.trainer_callback import EarlyStoppingCallback

import optuna
import evaluate

torch.backends.cudnn.benchmark = True
torch.use_deterministic_algorithms(False)

torch.autograd.profiler.profile(False)
torch.autograd.profiler.emit_nvtx(False)
torch.autograd.set_detect_anomaly(False)

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


 Загрузим датасет [Датасет](https://huggingface.co/datasets/voorhs/conll2003-corrupted).

[Загрузка с huggingface](https://huggingface.co/docs/datasets/load_hub).


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("voorhs/conll2003-corrupted")

In [ ]:
dataset['train'][0]


{'id': '473',
 'tokens': ['LONDON', '1996-08-22'],
 'pos_tags': [22, 11],
 'chunk_tags': [11, 12],
 'ner_tags': ['B-LOC', 'O']}

Мы будем предсказывать NER tags непосредственно по тексту. Остальная информация нам не понадобится. Удалите все поля кроме `tokens` и `ner_tags`.



In [ ]:
columns_to_keep = ['tokens', 'ner_tags']

# Для каждого набора данных (train, validation, test) удаляем лишние колонки
for split in dataset.keys():
    # Получаем все колонки текущего набора данных
    all_columns = dataset[split].column_names

    # Определяем колонки для удаления (все, кроме тех, что нужно сохранить)
    columns_to_remove = [col for col in all_columns if col not in columns_to_keep]

    # Удаляем ненужные колонки
    dataset[split] = dataset[split].remove_columns(columns_to_remove)

# Проверяем результат
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})


### BIO-разметка

**BIO-разметка** — это один из наиболее распространённых способов аннотации текста в задаче Named Entity Recognition (NER). Она используется для обозначения границ и типов именованных сущностей на уровне отдельных токенов.

В основе BIO-разметки лежат три типа тегов:

- B (Beginning) — указывает на начало именованной сущности;

- I (Inside) — обозначает продолжение той же сущности;

- O (Outside) — означает, что токен не относится ни к одной сущности.

Каждому токену в тексте присваивается один из BIO-тегов, часто дополненный типом сущности (например, PER, LOC, ORG):

B-PER — начало сущности типа «человек»;

I-PER — продолжение сущности типа «человек»;

O — токен вне любой сущности.

Такой формат помогает моделям точно определять границы и типы сущностей.

In [ ]:
# посмотрим какие NER-теги присутствуют в данных
# собираем все уникальные теги
all_tags = []

for split in dataset.values():
    # Получаем все ner_tags для текущего сплита
    split_tags = split['ner_tags']

    # "Разворачиваем" вложенные списки и добавляем в общий список
    for tag_list in split_tags:
        all_tags.extend(tag_list)

# Преобразуем в множество для получения уникальных значений
unique_tags = set(all_tags)

# Выводим результат
print(f"Всего уникальных NER-тегов: {len(unique_tags)}")
print("Уникальные NER-теги:", sorted(unique_tags))

Всего уникальных NER-тегов: 10
Уникальные NER-теги: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'MISSING', 'O']


### Маппинг тегов

Большинство моделей работают только с числами, а не со строками. Поэтому перед обучением или инференсом:

- Строковые теги переводятся в числа с помощью `label2id`.

- После предсказания моделью — наоборот, числа преобразуются обратно в строки с помощью `id2label`.



In [ ]:
# будем испльзовать регулярные выражения
import re

# 1. Собираем ВСЕ уникальные теги из датасета
def collect_all_tags(dataset):
    """Собирает все уникальные NER-теги из всех сплитов"""
    all_tags = []
    for dataset in dataset.values():
        for example in dataset:
            all_tags.extend(example['ner_tags'])
    return set(all_tags)

# Получаем все теги
all_unique_tags = collect_all_tags(dataset)
print("Все теги в датасете (включая подозрительные):")
print(sorted(all_unique_tags))
print(f"Всего: {len(all_unique_tags)}\n")

# 2. Определяем функцию для проверки корректности тегов
def is_valid_ner_tag(tag):
    """
    Проверяет, является ли тег корректным NER-тегом.
    Корректные форматы: O, B-XXX, I-XXX, B_XXX, I_XXX (где XXX - тип сущности)
    """
    if not isinstance(tag, str):
        return False

    # Подозрительные теги, которые нужно исключить
    suspicious_patterns = [
        '<MISSING>', '<missing>', 'MISSING', 'missing',
        'UNK', 'UNKNOWN', 'unknown',
        'NA', 'N/A', 'NONE', 'None', 'none',
        '',  # пустая строка
    ]

    # Проверяем, не является ли тег подозрительным
    if tag in suspicious_patterns:
        return False

    # Проверяем форматы NER-тегов
    # 1. Тег "O" (Outside - не сущность)
    if tag == 'O':
        return True

    # 2. Теги BIO-формата: B-XXX, I-XXX
    if re.match(r'^[BIES]-[A-Z_]+$', tag):
        return True

    # 3. Теги с подчеркиванием вместо дефиса: B_XXX, I_XXX
    if re.match(r'^[BIES]_[A-Z_]+$', tag):
        return True

    # 4. Некоторые датасеты используют числовые теги, преобразованные в строки
    # Проверяем, что тег не содержит недопустимых символов
    if re.match(r'^[A-Z0-9_-]+$', tag):
        # Исключаем чисто числовые теги (возможно, это неправильно закодированные)
        if tag.isdigit():
            return False
        return True

    return False

# 5. Фильтруем теги, оставляя только корректные
valid_tags = sorted([tag for tag in all_unique_tags if is_valid_ner_tag(tag)])
invalid_tags = sorted([tag for tag in all_unique_tags if not is_valid_ner_tag(tag)])

print("Корректные NER-теги:")
for tag in valid_tags:
    print(f"  - {tag}")
print(f"Всего корректных: {len(valid_tags)}\n")

print("Подозрительные/некорректные теги (будут исключены):")
for tag in invalid_tags:
    print(f"  - {tag}")
print(f"Всего некорректных: {len(invalid_tags)}\n")

# 6. Создаем маппинги только для корректных тегов
# Добавляем тег 'O' первым (если он есть), так как обычно он имеет индекс 0
label2id = {}
id2label = {}

# Если есть тег 'O', добавляем его первым
if 'O' in valid_tags:
    label2id['O'] = 0
    id2label[0] = 'O'

# Добавляем остальные теги в алфавитном порядке, но без 'O'
other_tags = [tag for tag in valid_tags if tag != 'O']
for idx, tag in enumerate(other_tags, start=len(label2id)):
    label2id[tag] = idx
    id2label[idx] = tag

# 7. Выводим созданные маппинги
print("=" * 60)
print("СОЗДАННЫЕ МАППИНГИ:")
print("=" * 60)

print("\nlabel2id (тег -> индекс):")
for tag, idx in label2id.items():
    print(f"  '{tag}': {idx}")

print("\nid2label (индекс -> тег):")
for idx, tag in sorted(id2label.items()):
    print(f"  {idx}: '{tag}'")

print(f"\nВсего классов в маппингах: {len(label2id)}")


# 8. Функция для очистки датасета от некорректных тегов
def clean_dataset_tags(dataset_dict, label2id):
    """Заменяет некорректные теги на 'O' и преобразует в числовые индексы"""

    def clean_example(example):
        # Заменяем некорректные теги на 'O'
        cleaned_tags = []
        for tag in example['ner_tags']:
            if tag in label2id:
                cleaned_tags.append(tag)
            else:
                cleaned_tags.append('O')  # заменяем некорректный тег на 'O'

        # Преобразуем теги в числовые индексы
        numeric_tags = [label2id[tag] for tag in cleaned_tags]

        return {
            'tokens': example['tokens'],
            'ner_tags': cleaned_tags,  # строковые теги
            'ner_tags_ids': numeric_tags  # числовые индексы
        }

    # Применяем очистку ко всем датасетам
    cleaned_datasets = {}
    for split_name, dataset in dataset_dict.items():
        cleaned_dataset = dataset.map(clean_example)
        cleaned_datasets[split_name] = cleaned_dataset

    return cleaned_datasets

# 9. Создаем очищенный датасет
if label2id:  # только если есть корректные теги
    cleaned_dataset_dict = clean_dataset_tags(dataset, label2id)


    # Проверяем, что все теги теперь корректны
    all_tags_after_clean = collect_all_tags(cleaned_dataset_dict)
    print(f"Теги после очистки: {sorted(all_tags_after_clean)}")
    print(f"Всего уникальных тегов после очистки: {len(all_tags_after_clean)}")

    # Пример очищенного примера
    print("\nПример очищенного примера (первые 2 токена):")
    example = cleaned_dataset_dict['train'][0]
    for i in range(min(2, len(example['tokens']))):
        print(f"  '{example['tokens'][i]}' -> {example['ner_tags'][i]} (id: {example['ner_tags_ids'][i]})")

Все теги в датасете (включая подозрительные):
['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'MISSING', 'O']
Всего: 10

Корректные NER-теги:
  - B-LOC
  - B-MISC
  - B-ORG
  - B-PER
  - I-LOC
  - I-MISC
  - I-ORG
  - I-PER
  - O
Всего корректных: 9

Подозрительные/некорректные теги (будут исключены):
  - MISSING
Всего некорректных: 1

СОЗДАННЫЕ МАППИНГИ:

label2id (тег -> индекс):
  'O': 0
  'B-LOC': 1
  'B-MISC': 2
  'B-ORG': 3
  'B-PER': 4
  'I-LOC': 5
  'I-MISC': 6
  'I-ORG': 7
  'I-PER': 8

id2label (индекс -> тег):
  0: 'O'
  1: 'B-LOC'
  2: 'B-MISC'
  3: 'B-ORG'
  4: 'B-PER'
  5: 'I-LOC'
  6: 'I-MISC'
  7: 'I-ORG'
  8: 'I-PER'

Всего классов в маппингах: 9


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Теги после очистки: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Всего уникальных тегов после очистки: 9

Пример очищенного примера (первые 2 токена):
  'LONDON' -> B-LOC (id: 1)
  '1996-08-22' -> O (id: 0)


### Фильтрация

В обучающей выборке для части объектов пропущена разметка.
Разделим обучающую выборку на размеченные и неразмеченные объекты.

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})


In [ ]:
def has_bio_tags(example):
    return any(tag.startswith(("B-", "I-")) for tag in example["ner_tags"])

def only_O_tags(example):
    return all(tag == "O" for tag in example["ner_tags"])


In [ ]:
labeled = dataset["train"].filter(has_bio_tags)
unlabeled = dataset["train"].filter(only_O_tags)

print(len(labeled), len(unlabeled))


Filter:   0%|          | 0/14041 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14041 [00:00<?, ? examples/s]

7936 2064


In [ ]:
# обновим датасет
from datasets import DatasetDict

dataset = DatasetDict({
    "train": labeled,
    "train_missing": unlabeled,
    "validation": dataset["validation"],
    "test": dataset["test"]
})


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 7936
    })
    train_missing: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2064
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

### Token-independent представление


*BIO-разметка* применяется к конкретным токенам, которые были получены с помощью определённого токенизатора (например, BERT, RoBERTa и т.д.).

У разных токенизаторов одни и те же слова могут разбиваться по-разному (например, на сабтокены).

BIO-теги, связанные с токенами старого токенизатора, могут не соответствовать токенам нового.

Поэтому, чтобы использовать любой другой токенизатор, нам нужно сделать универсальное представление сущностей, независящее от токенов.

**1. Восстановление текста**: Для каждого объекта датасета необходимо объединить токены из `tokens` в единую строку и поместить в поле `text`. Необходимо учесть, что знаки препинания, кавычки и скобки (например, `,`, `.`, `!`. `)`, `"`) не должны иметь пробелов перед собой, но должны иметь пробел после себя.


**2.** По имеющейся BIO-разметке необходимо:
* Оставить только последовательность сущностей, удалив префиксы начала (B- и I-).
* Для каждой именованной сущности найти позиции символов, ей соответствующие (в формате python slice).

Пример результата:
```python
{
    'text': '6 - Andre Agassi (U.S.) vs. Mauricio Hadad (Colombia)',
    'tags': ['PER', 'LOC', 'PER','LOC'],
    'ranges': [[4, 16], [19, 23], [30, 44], [47, 55]]
}
```

*Пояснение к примеру*. В тексте `'6 - Andre Agassi (U.S.) vs. Mauricio Hadad (Colombia)'` встретилось четыре сущности:
- `PER` на позициях `[4,16]`, а именно `Andre Agassi`
- `LOC` на позициях `[19,23]`, а именно `U.S.`
- `PER` на позициях `[30,44]`, а именно `Mauricio Hadad`
- `LOC` на позициях `[47,55]`, а именно `Colombia`

In [ ]:
def transform_ner_data(tokens: list[str], ner_tags: list[str]):
    """
    Obtain tokenizer-independent representation of a single sample.

    Args:
        tokens: list of words, subwords or even punctuations
        ner_tags: list of BIO labels
    Returns:
        sample: dictionary with the following contents:
            - text (str): detokenized sequence
            - tags (list[str]): list of tags names (PER, ORG, LOC, MISC)
            - ranges (list[tuple[int, int]]): list of python slices for each entity
    """

    punctuation_symbols = ("-", "'", ".", ",", ")", ":", ";", '"')
    text = ""
    ranges = []
    tags = []

    current_tag = None
    current_start = None

    for token, tag in zip(tokens, ner_tags):
        # Add space unless punctuation or beginning of text
        if len(text) > 0 and not token.startswith(punctuation_symbols):
            text += " "

        token_start = len(text)
        text += token
        token_end = len(text)

        # --- Handle NER tag ---
        if tag.startswith("B-"):
            # Close previous entity if it exists
            if current_tag is not None:
                ranges.append([current_start, prev_token_end])
                tags.append(current_tag)

            current_tag = tag[2:]  # remove B- prefix
            current_start = token_start

        elif tag.startswith("I-") and current_tag == tag[2:]:
            # Continue current entity
            pass  # just extend

        else:
            # Close previous entity if any
            if current_tag is not None:
                ranges.append([current_start, prev_token_end])
                tags.append(current_tag)
                current_tag = None
                current_start = None

        prev_token_end = token_end  # track end of last token

    # Final entity close if any
    if current_tag is not None:
        ranges.append([current_start, token_end])
        tags.append(current_tag)

    return {"text": text, "tags": tags, "ranges": ranges}


In [ ]:
dataset_generic = dataset.map(
    function=transform_ner_data,
    input_columns=["tokens", "ner_tags"],
    remove_columns=dataset["train"].column_names,
)

dataset_generic

Map:   0%|          | 0/7936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2064 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'tags', 'ranges'],
        num_rows: 7936
    })
    train_missing: Dataset({
        features: ['text', 'tags', 'ranges'],
        num_rows: 2064
    })
    validation: Dataset({
        features: ['text', 'tags', 'ranges'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['text', 'tags', 'ranges'],
        num_rows: 3453
    })
})

### Токенизация

Воспользуемся токенизатором модели `google-bert/bert-base-uncased`.

In [ ]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print(bert_tokenizer(dataset_generic["train"]["text"][11]))

{'input_ids': [101, 2707, 2322, 1017, 1017, 2403, 2656, 5179, 2260, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


У каждого токенизатора есть свои специальные токены, посмотрим какие.

In [ ]:
bert_tokenizer.all_special_ids

[100, 102, 0, 101, 103]

Теперь необходимо реализовать функцию, которая на основе уже подготовленного `tokenizer‑independent` представления сможет формировать BIO‑разметку для произвольного токенизатора.

Функция `align_tokens_and_tags`будет осуществлять преобразование одного элемента датасета в BIO‑размеченный вид, соответствующий выбранному токенизатору.

In [ ]:
SPECIAL_TOKENS_LABEL = -100

def align_tokens_and_tags(
    text: str,
    tags: list[str],
    ranges: list[tuple[int, int]],
    tokenizer,
    label2id: dict,
):
"""
    Align the tokenizer-independent representation with a Hugging Face tokenizer.

    Args:
        text: original text.
        tags: list of entity tags (e.g., ["PER", "LOC"]).
        ranges: list of (start, end) character ranges for each entity.
        tokenizer: A Hugging Face tokenizer.
        label2id: dict mapping BIO tag strings to indices. If None, simple B/I/O labels will be used as strings.

    Returns:
        dict: A dictionary with `input_ids`, `labels`:
            - input_ids are token ids from tokenizer's vocabulary
            - labels are integer labels for BIO tags (from label2id mapping)
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        padding=False,
    )

    input_ids = encoding["input_ids"]
    offset_mapping = encoding["offset_mapping"]

    #
    labels = [label2id["O"]] * len(input_ids)

    for tag, (ent_start, ent_end) in zip(tags, ranges):
        first = True
        for i, (tok_start, tok_end) in enumerate(offset_mapping):

            # спец-токены
            if tok_start == tok_end == 0:
                labels[i] = SPECIAL_TOKENS_LABEL
                continue

            # нет пересечения
            if tok_end <= ent_start or tok_start >= ent_end:
                continue

            prefix = "B-" if first else "I-"
            labels[i] = label2id[f"{prefix}{tag}"]  # ← int
            first = False

    return {
        "input_ids": input_ids,
        "labels": labels,  # ← ТОЛЬКО int / -100
    }


In [ ]:

def tokenize_ner_dataset(
    dataset: Union[datasets.DatasetDict, datasets.Dataset], tokenizer: AutoTokenizer
):
    return dataset.map(
        function=align_tokens_and_tags,
        input_columns=["text", "tags", "ranges"],
        fn_kwargs={"tokenizer": tokenizer, "label2id": label2id},
        remove_columns=(
            dataset["train"] if "train" in dataset else dataset
        ).column_names,
    )

Выведем несколько примеров:

In [ ]:
for i in range(9, 12):
    sample = dataset_generic["train"][i]
    print("before tokenization:", sample)
    tokenized_sample = align_tokens_and_tags(**sample, tokenizer=bert_tokenizer)
    print("after tokenization:", tokenized_sample)
    print()

before tokenization: {'text': 'The Vermonter, which runs between St. Albans, Vermont, near the Canadian border and Washington, D.C., collided with the truck at 7:51 a.m. EDT near the rural town of Roxbury some 15 miles southeast of the state capital Montpelier, Amtrak spokeswoman Maureen Garrity said.', 'tags': ['MISC', 'LOC', 'LOC', 'MISC', 'LOC', 'LOC', 'LOC', 'ORG', 'PER'], 'ranges': [[0, 13], [34, 44], [46, 53], [64, 72], [84, 100], [166, 173], [219, 229], [231, 237], [250, 265]]}
after tokenization: {'input_ids': [101, 1996, 8839, 2121, 1010, 2029, 3216, 2090, 2358, 1012, 26311, 1010, 8839, 1010, 2379, 1996, 3010, 3675, 1998, 2899, 1010, 1040, 1012, 1039, 1012, 1010, 17745, 2007, 1996, 4744, 2012, 1021, 1024, 4868, 1037, 1012, 1049, 1012, 3968, 2102, 2379, 1996, 3541, 2237, 1997, 20996, 2595, 4917, 2070, 2321, 2661, 4643, 1997, 1996, 2110, 3007, 18318, 11880, 3771, 1010, 20208, 3764, 26760, 20778, 19167, 11721, 18752, 3723, 2056, 1012, 102], 'labels': [-100, 'B-MISC', 'I-MISC', 'I

Тесты:

In [ ]:
dataset_tokenized = tokenize_ner_dataset(dataset_generic, bert_tokenizer)

Map:   0%|          | 0/7936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2064 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

##  Разметка с помощью LLM



`Qwen2.5` — это серия мощных открытых языковых моделей, разработанных компанией Alibaba. Они поддерживают различные размеры (от 0.5B до 72B параметров)

Qwen2.5 часто используется в задачах генерации текста, диалога и RAG-сценариях.

In [ ]:
def messages_maker(text: str) -> list[dict[str, str]]:
    """
    Creates a prompt for the LLM inference to generate NER labels.

    Args:
        text: The input text to be labeled.
    Returns:
        list: A list of messages formatted for the LLM inference.
    """
    system_message = (
        "You are an expert in Named Entity Recognition (NER). Your task is to analyze the given text and identify "
        "entities such as persons (PER), locations (LOC), organizations (ORG), and miscellaneous (MISC). "
        "You must return the results strictly in JSON format with the following fields:\n\n"
        "1. 'tags': A list of entity type strings (e.g., ['PER', 'LOC', 'ORG', 'PER']). Each tag corresponds to one detected entity.\n"
        "2. 'ranges': A list of character spans (start and end indices) for each entity, formatted as [start, end]. The indices must match character positions in the original input text.\n"
        "3. 'reasoning': A free-text explanation of how and why you chose the entities and their types — explain your reasoning step by step.\n\n"
        "Important rules:\n"
        "- Do NOT return BIO labels. Just plain entity types (e.g., 'PER', 'LOC', etc.).\n"
        "- The number of 'tags' and 'ranges' must be equal.\n"
        "- Each span in 'ranges' must align exactly with an entity in the text.\n"
        "- Return only one JSON object. No extra explanations or preambles.\n"
    )

    user_message = f"Please analyze the following text and generate the NER labels as described:\n\n{text}"

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    return messages


### Работа с LLM в режиме `chat mode`

[Инструкция](https://huggingface.co/docs/transformers/main/en/conversations#textgenerationpipeline) как загружать модели `chat mode`
 в Transformers из командной строки, как создавать и форматировать беседу, а также как общаться в чате с помощью `textgenerationpipeline`

In [ ]:

def generate_answer(generator: TextGenerationPipeline, query: str) -> str:
     chat = generator(messages_maker(query), max_new_tokens=1024)[0]["generated_text"]
     return chat[-1]["content"]

Эта функция отправляет текстовый запрос `query` большой языковой модели и возвращает ответ:

In [ ]:
from transformers import pipeline


model_name = "Qwen/Qwen2.5-7B-Instruct"

generator = pipeline(model=model_name, task="text-generation", device="cuda")
print(type(generator))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda


<class 'transformers.pipelines.text_generation.TextGenerationPipeline'>


In [ ]:
print_params_count(generator.model)

Информация о числе параметров модели: Qwen2ForCausalLM
Всего параметров: 		 7615616512
Всего обучаемых параметров: 	 7615616512



In [ ]:
generate_answer(generator, "I live in London, but my father is from Germany.")

'{\n  "tags": ["LOC", "LOC", "PER", "LOC"],\n  "ranges": [[8, 15], [22, 29], [26, 30], [37, 43]],\n  "reasoning": "The phrase \'London\' is identified as a location (LOC) as it refers to a city. Similarly, \'Germany\' is also recognized as a location (LOC) since it is the name of a country. \'My father\' is considered a person (PER) as it refers to a family member. Since we don\'t have the full name, we can only identify the relation rather than the specific individual."\n}'

In [ ]:
generate_answer(generator, "I live in London, but my father is from Germany.")

'{\n  "tags": ["LOC", "LOC", "PER"],\n  "ranges": [[8, 15], [26, 33], [29, 34]],\n  "reasoning": "The phrase \'in London\' indicates a location, so \'London\' is tagged as LOC. Similarly, \'from Germany\' also refers to a location, thus \'Germany\' is also tagged as LOC. The word \'father\' implies a person, hence it is tagged as PER."\n}'

In [ ]:
generate_answer(generator, "Can't wait the Oasis reunion.")

'{\n  "tags": ["ORG", "PER"],\n  "ranges": [[10, 15], [0, 9]],\n  "reasoning": "The text mentions \'Oasis\' which is a well-known band, thus it is categorized as an organization (ORG). \'Oasis\' reunion refers to the band\'s comeback, so the word \'Can\'t wait\' before it implies a person\'s anticipation, hence \'Can\'t wait\' is considered as a reference to a person (PER)."\n}'

In [ ]:
generate_answer(generator, "Check, please.")

'{\n  "tags": [\n    "MISC"\n  ],\n  "ranges": [\n    [\n      7,\n      11\n    ]\n  ],\n  "reasoning": "The word \'check\' in this context is likely a request for the bill at a restaurant, which falls under the category of a miscellaneous entity (MISC). It does not refer to a specific person, location, or organization."\n}'

### Обработка ответа LLM

Напишем функцию `generate_ner_labeling`, которая для данного примера:
- обращается к LLM чтобы получить разметку (с помощью функции `generate_answer`, определенной ранее)
- парсит ответ LLM в питоновский словарь (с помощью функции `parse`, см. ниже)
- валидирует словарь (с помощью функции `validate`, определенной ранее)
- делает `k_tries` попыток, пока LLM не выдаст валидный ответ



In [ ]:
def generate_ner_labeling(
    text: str,
    generator: TextGenerationPipeline,
    k_tries: int = 3
) -> dict:
    """
    Generate NER annotation using LLM with retries.

    Returns:
        dict with fields 'text', 'tags', 'ranges', 'reasoning', or None-filled fallback.
    """
    for attempt in range(k_tries):
        try:
            response = generate_answer(generator, text)
            parsed = parse(response)

            # Дополнительная валидация (можно расширить)
            if not isinstance(parsed.get("tags"), list) or not isinstance(parsed.get("ranges"), list):
                raise ValueError("Missing or invalid 'tags' or 'ranges' fields.")

            if len(parsed["tags"]) != len(parsed["ranges"]):
                raise ValueError("Mismatch between number of tags and ranges.")

            # Обязательно вставляем исходный текст
            parsed["text"] = text

            print("Successful generation!")
            return parsed

        except (json.JSONDecodeError, ValueError, KeyError) as e:
            print(f"[Attempt {attempt+1}] Invalid response: {e}\nResponse: {locals().get('response', '')}\nRetrying...\n")
            continue

        finally:
            gc.collect()
            torch.cuda.empty_cache()

    # Если ни одна попытка не удалась
    print(f"Failed to generate valid NER labeling after {k_tries} attempts.")
    return {
        "text": text,
        "tags": None,
        "ranges": None,
        "reasoning": None
    }













In [ ]:
import gc
import torch
import json
from typing import Any
from transformers import TextGenerationPipeline


def parse(response: str) -> dict[str, Any]:
    """
    Parse LLM's response into dictionary.
    """
    start_dict = response.find("{")
    end_dict = len(response) - response[::-1].find("}") - 1
    return json.loads(response[start_dict : end_dict + 1])

In [ ]:
generate_answer(generator, "I live in London but my dog Jack doesn't like it.")

' ```json\n{\n  "tags": ["LOC", "MISC"],\n  "ranges": [[9, 15], [23, 26]],\n  "reasoning": "London is a location, so it is classified as LOC. Jack is a name given to a dog, which falls under the Miscellaneous category, hence MISC."\n}\n```'

Хороший результат должен выглядеть подобным образом:

In [ ]:
result = generate_ner_labeling(
    "I live in London but my dog Jack doesn't like it.",
    generator
)

Successful generation!


In [ ]:
generate_ner_labeling("I live in London but my dog Jack doesn't like it.", generator)

Successful generation!


{'tags': ['LOC', 'PER'],
 'ranges': [[8, 15], [21, 24]],
 'reasoning': "In the text, 'London' is identified as a location (LOC) because it refers to a city. 'Jack' is identified as a person (PER) since it is used as a name for the dog.",
 'text': "I live in London but my dog Jack doesn't like it."}

Загрузите из памяти ранее рагруженный датасет в tokenizer-independent представлении.

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 7936
    })
    train_missing: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2064
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

В качестве демонстрации разметим только 10 примеров в пределах от 15 до 30 символов, для экономии ресурсов:

In [ ]:
from datasets import Dataset
from copy import deepcopy

# Список для сохранения отобранных примеров
filtered = []

# Перебираем примеры по индексам
for i in range(3, 100):  # увеличен диапазон, чтобы точно набрать 10
    sample = dataset_generic["train_missing"][i]

    if 15 <= len(sample["text"]) <= 40:
        filtered.append(deepcopy({
            "text": sample["text"],
            "tags": sample["tags"],
            "ranges": sample["ranges"]
        }))

    if len(filtered) == 10:
        break

# Создаем независимый Dataset
subset_dataset = Dataset.from_list(filtered)

# Выводим результат
print(subset_dataset)
print(subset_dataset['text'])


Dataset({
    features: ['text', 'tags', 'ranges'],
    num_rows: 10
})
Column(['MONDAY, AUGUST 26TH SCHEDULE', 'CLOSE IN POINTS', '1998 665M 840M 570M', "Women's singles", 'Scoreboard on the fourth day of'])


In [ ]:
synthetic = [generate_ner_labeling(sample["text"], generator) for sample in subset_dataset]

Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!
Successful generation!


In [ ]:
synthetic

[{'tags': ['PER', 'LOC'],
  'ranges': [[0, 5], [14, 28]],
  'reasoning': "The text mentions 'MONDAY, AUGUST 26TH', which could refer to a specific day and month, but since no specific person's name is provided, it's not clear if 'MONDAY' is meant as a person. However, 'AUGUST 26TH' can be considered a date, which doesn't fit neatly into any of the given categories. Given the instruction, 'MONDAY' is treated as PER due to its positional importance in scheduling, while 'AUGUST 26TH' is left out as it does not clearly fit any category. The 'SCHEDULE' at the end is a general term and does not represent a specific location, organization, or person, so it is omitted from the tags and ranges.",
  'text': 'MONDAY, AUGUST 26TH SCHEDULE'},
 {'tags': [],
  'ranges': [],
  'reasoning': "The text 'CLOSE IN POINTS' does not contain any clear references to persons, locations, organizations, or any other named entities. Therefore, no entities are identified.",
  'text': 'CLOSE IN POINTS'},
 {'tags': [